In [1]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertConfig,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split

In [2]:
DATA_PATH = "../data/processed/outcome_dataset.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,text,label
0,War breaks out between two countries,HIGH
1,Airstrike kills 25 civilians,HIGH
2,Terrorist attack reported in city,HIGH
3,Border tensions escalate between nations,HIGH
4,Explosion injures dozens in market,HIGH


In [3]:
#label mapping

label_map = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
df["label"] = df["label"].map(label_map)

df.head()

,text,label
0,War breaks out between two countries,2
1,Airstrike kills 25 civilians,2
2,Terrorist attack reported in city,2
3,Border tensions escalate between nations,2
4,Explosion injures dozens in market,2


In [4]:
#spilliting the data in train & test 

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [5]:
#Tokenixer 

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

In [6]:
#dataset class 

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)

In [7]:
#load model 

config = DistilBertConfig.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    problem_type="single_label_classification"
)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    config=config
)

/home/mayankk/Deep Learning/torch_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
#traning config

training_args = TrainingArguments(
    output_dir="../models/outcome_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    learning_rate=5e-5,
    load_best_model_at_end=True
)

/home/mayankk/Deep Learning/torch_env/lib/python3.12/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [9]:
#trainer 

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [10]:
#train
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.840980
2,No log,0.622850
3,No log,0.533571


TrainOutput(global_step=24, training_loss=0.7931429545084635, metrics={'train_runtime': 5.2251, 'train_samples_per_second': 34.449, 'train_steps_per_second': 4.593, 'total_flos': 419142603240.0, 'train_loss': 0.7931429545084635, 'epoch': 3.0})

In [12]:
trainer.save_model("../models/outcome_model")
tokenizer.save_pretrained("../models/outcome_model")

('../models/outcome_model/tokenizer_config.json',
 '../models/outcome_model/special_tokens_map.json',
 '../models/outcome_model/vocab.txt',
 '../models/outcome_model/added_tokens.json',
 '../models/outcome_model/tokenizer.json')